# 07 - Análise Temporal: CNEFE 2010 × Censo 2022 (BA)

Compara o perfil de cada setor censitário entre os dois momentos.

**CNEFE 2010** não tem COD_ESPECIE, mas tem:
- `total_enderecos` por setor (proxy de densidade)
- `situacao` 1=urbano, 2=rural/localidade (proxy de urbanidade)

**O que podemos mapear:**
- Crescimento/declínio de endereços por setor
- Setores novos em 2022 (expansão urbana)
- Setores que mudaram de perfil rural → urbano
- Correlação entre crescimento e tipo de cluster em 2022

In [ ]:
import duckdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

con = duckdb.connect()
CNEFE_2010 = "'../data/cnefe_2010/*.snappy.parquet'"
OUTPUT_DIR = Path('../outputs')

In [ ]:
# Perfil 2010 por setor (filtrando BA)
df_2010 = con.execute(f"""
    SELECT
        setor                                                      AS cod_setor,
        COUNT(*)                                                   AS total_2010,
        SUM(CASE WHEN situacao = '1' THEN 1 ELSE 0 END) * 1.0
            / COUNT(*)                                             AS prop_urbano_2010,
        SUM(CASE WHEN situacao = '2' THEN 1 ELSE 0 END) * 1.0
            / COUNT(*)                                             AS prop_rural_2010
    FROM read_parquet({CNEFE_2010})
    WHERE uf = '29'
    GROUP BY setor
""").df()

print(f'Setores BA em 2010: {len(df_2010):,}')
df_2010.head()

In [ ]:
# Carrega perfil 2022 com clusters já atribuídos
df_2022 = pd.read_parquet(OUTPUT_DIR / 'setores_clusterizados.parquet')
df_2022 = df_2022.rename(columns={
    'COD_SETOR': 'cod_setor',
    'total_enderecos': 'total_2022'
})

print(f'Setores BA em 2022: {len(df_2022):,}')
df_2022[['cod_setor', 'total_2022', 'lat_centroide', 'lon_centroide', 'cluster', 'cluster_nome']].head()

In [ ]:
# Join: setores presentes nos dois anos
df = df_2022[['cod_setor', 'total_2022', 'lat_centroide', 'lon_centroide',
              'cluster', 'cluster_nome']].merge(
    df_2010[['cod_setor', 'total_2010', 'prop_urbano_2010', 'prop_rural_2010']],
    on='cod_setor', how='left'
)

novos_2022   = df['total_2010'].isna().sum()
em_ambos     = df['total_2010'].notna().sum()
apenas_2010  = len(df_2010) - em_ambos

print(f'Setores presentes nos dois anos : {em_ambos:,}')
print(f'Setores novos em 2022           : {novos_2022:,}  (expansão urbana)')
print(f'Setores apenas em 2010          : {apenas_2010:,}  (extintos ou reclassificados)')

In [ ]:
# Métricas de mudança para setores em ambos os anos
df_match = df[df['total_2010'].notna()].copy()

df_match['delta_abs']  = df_match['total_2022'] - df_match['total_2010']
df_match['taxa_cresc'] = (df_match['total_2022'] - df_match['total_2010']) / df_match['total_2010']

print(df_match[['total_2010', 'total_2022', 'delta_abs', 'taxa_cresc']].describe().round(2))

In [ ]:
# Classificação da trajetória de cada setor
def classificar_trajetoria(row):
    if row['prop_rural_2010'] > 0.5 and row['taxa_cresc'] > 0.3:
        return 'Rural → Urbano'
    if row['prop_urbano_2010'] > 0.8 and row['taxa_cresc'] > 0.3:
        return 'Adensamento Urbano'
    if row['taxa_cresc'] < -0.2:
        return 'Declínio'
    if abs(row['taxa_cresc']) <= 0.2:
        return 'Estável'
    return 'Crescimento Moderado'

df_match['trajetoria'] = df_match.apply(classificar_trajetoria, axis=1)

# Setores novos em 2022
df_novos = df[df['total_2010'].isna()].copy()
df_novos['trajetoria'] = 'Novo Setor (2022)'
df_novos['delta_abs']  = np.nan
df_novos['taxa_cresc'] = np.nan

df_tempo = pd.concat([df_match, df_novos], ignore_index=True)

print('Distribuição de trajetórias:')
print(df_tempo['trajetoria'].value_counts())

In [ ]:
# Distribuição da taxa de crescimento por cluster de 2022
fig, ax = plt.subplots(figsize=(13, 5))
ordem = df_match.groupby('cluster_nome')['taxa_cresc'].median().sort_values(ascending=False).index
sns.boxplot(data=df_match[df_match['cluster_nome'] != 'Ruído'],
            x='cluster_nome', y='taxa_cresc', order=ordem, ax=ax)
ax.axhline(0, color='red', linestyle='--', linewidth=0.8)
ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha='right', fontsize=8)
ax.set_title('Taxa de crescimento 2010→2022 por cluster')
ax.set_xlabel('')
ax.set_ylabel('Taxa de crescimento')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'figures/crescimento_por_cluster_ba.png', dpi=150)
plt.show()

In [ ]:
# Mapa: trajetória de cada setor
palette_traj = {
    'Rural → Urbano'     : '#e74c3c',
    'Adensamento Urbano' : '#e67e22',
    'Crescimento Moderado': '#f1c40f',
    'Estável'            : '#2ecc71',
    'Declínio'           : '#3498db',
    'Novo Setor (2022)'  : '#9b59b6',
}

fig, ax = plt.subplots(figsize=(10, 12))
for traj, cor in palette_traj.items():
    sub = df_tempo[df_tempo['trajetoria'] == traj]
    if len(sub):
        ax.scatter(sub['lon_centroide'], sub['lat_centroide'],
                   s=1, alpha=0.5, color=cor, label=f'{traj} (n={len(sub):,})')

ax.legend(markerscale=6, bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=8)
ax.set_title('Trajetória dos Setores 2010 → 2022 (BA)')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'figures/mapa_trajetoria_temporal_ba.png', dpi=150)
plt.show()

In [ ]:
# Cruzamento: trajetória × cluster 2022
pivot = (df_tempo[df_tempo['cluster_nome'] != 'Ruído']
         .groupby(['trajetoria', 'cluster_nome'])
         .size()
         .unstack(fill_value=0))

fig, ax = plt.subplots(figsize=(14, 5))
pivot.plot(kind='bar', ax=ax, colormap='tab20')
ax.set_title('Trajetória temporal × Cluster 2022')
ax.set_xlabel('Trajetória')
ax.set_ylabel('Número de setores')
ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=8)
plt.xticks(rotation=25, ha='right')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'figures/trajetoria_vs_cluster_ba.png', dpi=150)
plt.show()

In [ ]:
# Salvar
df_tempo.to_parquet(OUTPUT_DIR / 'analise_temporal_ba.parquet', index=False)
print('Salvo: outputs/analise_temporal_ba.parquet')